# Climate Downscaling Workflow (Detailed)

This notebook demonstrates an end-to-end deep learning downscaling template:
- synthetic climate-like data generation,
- CNN-based super-resolution,
- quantitative evaluation (MAE, RMSE, bias),
- visual diagnostics for research reporting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
class SyntheticClimateDataset(Dataset):
    def __init__(self, n=200, low_h=16, low_w=16, upscale=4, seed=0):
        rng = np.random.default_rng(seed)
        high_h, high_w = low_h * upscale, low_w * upscale
        yy, xx = np.meshgrid(np.linspace(0, 1, high_h), np.linspace(0, 1, high_w), indexing='ij')
        xs, ys = [], []
        for _ in range(n):
            phase = rng.uniform(0, 2*np.pi)
            high = (
                20
                + 2.5 * yy + 1.5 * xx
                + np.sin(2*np.pi*(xx + phase))
                + np.cos(2*np.pi*yy)
                + 0.35*np.sin(8*np.pi*xx)*np.cos(6*np.pi*yy)
                + rng.normal(0, 0.2, size=(high_h, high_w))
            )
            low = high.reshape(low_h, upscale, low_w, upscale).mean(axis=(1, 3))
            xs.append(low.astype(np.float32)[None, ...])
            ys.append(high.astype(np.float32)[None, ...])
        self.x = np.stack(xs)
        self.y = np.stack(ys)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return torch.from_numpy(self.x[idx]), torch.from_numpy(self.y[idx])

train_ds = SyntheticClimateDataset(n=240, seed=42)
val_ds = SyntheticClimateDataset(n=80, seed=43)
test_ds = SyntheticClimateDataset(n=80, seed=44)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)
len(train_ds), len(val_ds), len(test_ds)

In [ ]:
class DownscalerCNN(nn.Module):
    def __init__(self, upscale=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Upsample(scale_factor=upscale, mode='bilinear', align_corners=False),
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, x):
        return self.net(x)

model = DownscalerCNN(upscale=4).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

def eval_loss(loader):
    model.eval()
    vals = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            vals.append(loss_fn(model(x), y).item())
    return float(np.mean(vals))

history = {'train': [], 'val': []}
best_state = None
best_val = float('inf')
for epoch in range(1, 16):
    model.train()
    train_losses = []
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        opt.step()
        train_losses.append(loss.item())
    tr = float(np.mean(train_losses))
    va = eval_loss(val_loader)
    history['train'].append(tr)
    history['val'].append(va)
    if va < best_val:
        best_val = va
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    print(f'Epoch {epoch:02d} | train={tr:.4f} | val={va:.4f}')

model.load_state_dict(best_state)

In [ ]:
model.eval()
y_true_all, y_pred_all = [], []
with torch.no_grad():
    for x, y in test_loader:
        pred = model(x.to(device)).cpu().numpy().ravel()
        y_pred_all.append(pred)
        y_true_all.append(y.numpy().ravel())

y_true = np.concatenate(y_true_all)
y_pred = np.concatenate(y_pred_all)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
bias = np.mean(y_pred - y_true)
print({'MAE': float(mae), 'RMSE': float(rmse), 'Bias': float(bias)})

plt.figure(figsize=(6, 4))
plt.plot(history['train'], label='train')
plt.plot(history['val'], label='val')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Curve')
plt.legend()
plt.show()


In [ ]:
x_batch, y_batch = next(iter(test_loader))
with torch.no_grad():
    p_batch = model(x_batch.to(device)).cpu().numpy()

idx = 0
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].imshow(x_batch[idx, 0], cmap='coolwarm')
axes[0].set_title('Low-resolution input')
axes[1].imshow(p_batch[idx, 0], cmap='coolwarm')
axes[1].set_title('Predicted high-resolution')
axes[2].imshow(y_batch[idx, 0], cmap='coolwarm')
axes[2].set_title('Ground truth high-resolution')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


## Adapting to Real Climate Data

1. Replace `SyntheticClimateDataset` with an `xarray` loader for CMIP6 predictors and observed high-resolution targets.
2. Add temporal train/validation/test splits to avoid leakage.
3. Save prediction grids and compute region-wise metrics.
4. Quantify uncertainty (ensembles, bootstrap, or probabilistic models).
5. Compare against baseline methods (e.g., bilinear interpolation, random forest, and dynamical/statistical references).
